# الدرس 06: ربط واستخدام نماذج اللغات فائقة السرعة مع Groq API
### Fast LLM Inference with Groq Cloud & LangChain Integration (Lecture 31)

في هذا الدرس نطبق الآتي:
- إعداد مفتاح ومكتبات Groq في بايثون.
- استدعاء النماذج عبر `ChatGroq` والاستدعاء الموحد `init_chat_model`.
- فحص النماذج النشطة وتجربة البث اللحظي (Streaming).
- ربط نموذج Groq بخط أنابيب RAG مع ChromaDB.

### 1. استيراد المكتبات والتحقق من مفتاح البيئة (`GROQ_API_KEY`)
**آلية العمل:**
نقوم بتحميل متغيرات البيئة من ملف `.env`، والتحقق من وجود المفتاح السري الذي أنشأناه من لوحة تحكم `console.groq.com` لضمان صحة الاتصال قبل إرسال أي طلبات.

In [ ]:
# import standard libraries and load environment variables
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    print("GROQ_API_KEY loaded successfully! (starts with:", groq_api_key[:8], "...)")
else:
    raise ValueError("GROQ_API_KEY is missing! Please add it to your .env file.")

### 2. الطريقة الأولى: الاستدعاء المباشر عبر `langchain_groq` (`ChatGroq`)
**آلية العمل:**
هذه هي الطريقة المخصصة والأكثر شيوعاً، حيث نستخدم كلاس `ChatGroq` مباشرة ونحدد:
- اسم النموذج (`model`).
- درجة العشوائية (`temperature`): قيمة منخفضة (0.1 - 0.3) للإجابات الدقيقة والحقائق.
- الحد الأقصى للكلمات المولدة (`max_tokens`).

In [ ]:
# method 1: direct provider instantiation via langchain-groq
from langchain_groq import ChatGroq

llm_direct = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512,
    api_key=groq_api_key
)

# test direct invocation
prompt_text = "Explain what makes Groq LPUs so fast in two concise sentences."
response = llm_direct.invoke(prompt_text)

print(f"Prompt: {prompt_text}\n")
print(f"Response (via ChatGroq):\n{response.content}")

### 3. الطريقة الثانية: الواجهة القياسية الموحدة (`init_chat_model`)
**آلية العمل:**
في المشاريع الاحترافية ومشاريع الشركات، نفضل استخدام `init_chat_model` لأنها تفصل كود التطبيق عن مزود الموديل.
يمكنك التبديل لاحقاً بين Groq و OpenAI و Anthropic بمجرد تغيير معامل `model_provider` دون الحاجة لتعديل باقي الكود.

In [ ]:
# method 2: standard universal model initialization
from langchain.chat_models import init_chat_model

llm_unified = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.2,
    max_tokens=512
)

# test invocation with unified interface
prompt_unified = "In two concise sentences, what is the key difference between a CPU and a GPU?"
response_unified = llm_unified.invoke(prompt_unified)
print(f"Prompt: {prompt_unified}\n")
print(f"Response (via init_chat_model):\n{response_unified.content}")

### 4. استعراض النماذج المتاحة والنشطة على منصة Groq
**آلية العمل:**
تتحدث النماذج في المنصات السحابية باستمرار (يتم ترقية نماذج وتوقف أخرى قديمة كما حدث مع إصدارات Gemma القديمة).
يمكننا استخدام عميل `groq` الرسمي لسحب قائمة بكافة النماذج المتاحة لحسابك حالياً حتى تختار منها بسهولة.

In [ ]:
# list currently active models on Groq via python sdk
import groq

client = groq.Groq(api_key=groq_api_key)
available_models = [model.id for model in client.models.list().data]

print(f"Found {len(available_models)} active models on Groq:\n")
for idx, model_name in enumerate(sorted(available_models), start=1):
    print(f"  [{idx}] {model_name}")

### 5. البث المباشر للكلمات لحظة بلحظة (Real-Time Token Streaming)
**آلية العمل:**
واحدة من أقوى ميزات معالجات LPU في Groq هي السرعة الخارقة في توليد الكلمات.
باستخدام ميثود `.stream()`، لا ننتظر حتى ينتهي النموذج بالكامل من كتابة الرد، بل يتم طباعة كل كلمة فور خروجها من المعالج، مما يعطي تجربة مستخدم فورية تشبه ChatGPT.

In [ ]:
# test real-time streaming
stream_prompt = "Write 3 brief bullet points explaining why fast inference matters in AI chatbots."
print(f"Prompt: {stream_prompt}\n")
print("Streaming Response: ")

for chunk in llm_direct.stream(stream_prompt):
    print(chunk.content, end="", flush=True)
print("\n\n[Streaming Completed!]")

### 6. الدمج الكامل مع مسار الـ RAG وقاعدة بيانات المتجهات (ChromaDB)
**آلية العمل:**
الآن نربط نموذج Groq فائق السرعة مع قاعدة بيانات المتجهات المركزية التي بنيناها في الدروس السابقة داخل `data/databases/chroma_vector_store`.
نستخدم تعبير LCEL بمشغل الربط (`|`) لنشاهد كيف يسترجع النموذج السياق من ChromaDB ويولد الإجابة في أجزاء من الثانية!

In [ ]:
# connect to centralized ChromaDB vector store
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# define context-grounded prompt
template = """You are a concise AI assistant. Use ONLY the following retrieved context to answer the question.
Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# helper function
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# construct lcel rag chain with fast Groq LLM
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_direct
    | StrOutputParser()
)

# execute test query
rag_query = "What is computer vision?"
rag_answer = rag_chain.invoke(rag_query)

print(f"❓ Question: {rag_query}\n")
print(f"💬 Groq RAG Answer:\n{rag_answer}")

## 🎯 ملخص الدرس 06 (المحاضرة 31):
1. **Groq LPU**: بديل ممتاز ومجاني لنماذج OpenAI أثناء التعلم والتطوير، مع سرعات توليد مذهلة.
2. **طريقتان للاستدعاء**: `ChatGroq` للاستخدام المباشر والسريع، و `init_chat_model` للكود المعياري والحيادي لمزودي النماذج.
3. **التدفق اللحظي (`.stream()`)**: ميزة مدعومة أصلياً وتمنح تطبيقات RAG سرعة تفاعلية عالية.
4. **التوافق التام**: نموذج Groq يعمل فورياً مع سلاسل الـ RAG، والـ LCEL، وقواعد بيانات المتجهات بدون أي تعديلات إضافية.